In [2]:
# Step 1: Install kagglehub and download IMIDataset
!pip install -q kagglehub

import kagglehub
from pathlib import Path

print("Downloading IMIDataset...")
dataset_dir = kagglehub.dataset_download("patelp09/imidataset")
print(f"Dataset stored at: {dataset_dir}")

# Inspect available instrument folders
base_path = Path(dataset_dir)
folders = [p.name for p in base_path.iterdir() if p.is_dir()]
files = [p.name for p in base_path.iterdir() if p.is_file()]

print("\nSubdirectories found:", folders)
print("Files in root:", files)

Resuming download from 414187520 bytes (8464724695 bytes left)...
Resuming download to /root/.cache/kagglehub/datasets/patelp09/imidataset/1.archive (414187520/8878912215) bytes left.


100%|██████████| 8.27G/8.27G [07:46<00:00, 18.2MB/s]

Extracting files...


Dataset stored at: /root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1

Subdirectories found: ['Imid-06', 'Imid-05', 'Imid-04', 'Imid-03', 'Imid-02', 'Imid-01']
Files in root: []


In [3]:
# Step 2: Inspect subfolder contents and identify instrument mapping
from pathlib import Path

base_path = Path("/root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1")

# Look inside the first folder
first_part = base_path / "Imid-01"
print(f"Contents of {first_part.name}:")
sub_items = list(first_part.iterdir())[:15]
for item in sub_items:
    print(f"  {'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}")

# Check if any CSV/metadata files exist across the dataset
print("\nSearching for metadata files (.csv, .txt, .json):")
metadata_files = list(base_path.glob("**/*.csv")) + list(base_path.glob("**/*.txt")) + list(base_path.glob("**/*.json"))
print(metadata_files[:10])

Contents of Imid-01:
  [DIR]  dataset

Searching for metadata files (.csv, .txt, .json):
[]


In [4]:
# Step 3: Inspect inside Imid-01/dataset
from pathlib import Path

base_path = Path("/root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1")
dataset_dir = base_path / "Imid-01" / "dataset"

print(f"Checking: {dataset_dir}")
if dataset_dir.exists():
    items = list(dataset_dir.iterdir())
    print(f"Total items inside {dataset_dir.name}: {len(items)}")
    print("\nFirst 15 items:")
    for item in items[:15]:
        print(f"  {'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}")
else:
    print("Path does not exist; checking all dirs under Imid-01:")
    for p in (base_path / "Imid-01").glob("**/*"):
        if p.is_dir():
            print("  [DIR]", p.relative_to(base_path))

Checking: /root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1/Imid-01/dataset
Total items inside dataset: 2310

First 15 items:
  [FILE] violin_1109song.wav
  [FILE] flute_222song.wav
  [FILE] violin_285song.wav
  [FILE] piano_469song.wav
  [FILE] flute_608song.wav
  [FILE] guitar_417song.wav
  [FILE] guitar_360song.wav
  [FILE] Bass_56song.wav
  [FILE] piano_562song.wav
  [FILE] sarod_389song.wav
  [FILE] violin_1144song.wav
  [FILE] guitar_482song.wav
  [FILE] flute_756song.wav
  [FILE] guitar_339song.wav
  [FILE] flute_154song.wav


In [5]:
# Step 4: Index all audio files and check class distribution
from pathlib import Path
from collections import Counter

base_path = Path("/root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1")

# Target 10 instruments per course proposal
TARGET_CLASSES = {
    "flute", "veena", "sitar", "sarod", "tabla",
    "violin", "guitar", "piano", "bass", "drums"
}

all_wavs = list(base_path.glob("**/*.wav"))
print(f"Total .wav files found: {len(all_wavs)}")

# Extract class from filename prefix
class_counts = Counter()
matched_files = []

for wav in all_wavs:
    cls_name = wav.name.split("_")[0].lower()
    if cls_name in TARGET_CLASSES:
        class_counts[cls_name] += 1
        matched_files.append((wav, cls_name))

print(f"\nMatched target files: {len(matched_files)}")
print("\nDistribution across 10 target classes:")
for cls in sorted(TARGET_CLASSES):
    print(f"  {cls:10s}: {class_counts[cls]} tracks")

Total .wav files found: 9758

Matched target files: 9077

Distribution across 10 target classes:
  bass      : 1602 tracks
  drums     : 0 tracks
  flute     : 878 tracks
  guitar    : 798 tracks
  piano     : 1746 tracks
  sarod     : 554 tracks
  sitar     : 448 tracks
  tabla     : 644 tracks
  veena     : 636 tracks
  violin    : 1771 tracks


In [6]:
# Step 5: Inspect unmatched audio prefixes
from collections import Counter
from pathlib import Path

base_path = Path("/root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1")
all_wavs = list(base_path.glob("**/*.wav"))

TARGET_CLASSES = {
    "flute", "veena", "sitar", "sarod", "tabla",
    "violin", "guitar", "piano", "bass", "drums"
}

unmatched_prefixes = Counter()
for wav in all_wavs:
    prefix = wav.name.split("_")[0].lower()
    if prefix not in TARGET_CLASSES:
        unmatched_prefixes[prefix] += 1

print("Prefixes not in the 10 target classes:")
for prefix, count in unmatched_prefixes.most_common():
    print(f"  {prefix:15s}: {count} tracks")

Prefixes not in the 10 target classes:
  drum           : 681 tracks


In [7]:
# Step 6: Verify mapping of drum -> drums and check final dataset balance
from pathlib import Path
from collections import Counter

base_path = Path("/root/.cache/kagglehub/datasets/patelp09/imidataset/versions/1")
all_wavs = list(base_path.glob("**/*.wav"))

# 10 core instruments per course proposal
TARGET_MAP = {
    "flute": "flute",
    "veena": "veena",
    "sitar": "sitar",
    "sarod": "sarod",
    "tabla": "tabla",
    "violin": "violin",
    "guitar": "guitar",
    "piano": "piano",
    "bass": "bass",
    "drums": "drums",
    "drum": "drums",  # map singular to drums
}

final_counts = Counter()
indexed_samples = []

for wav in all_wavs:
    prefix = wav.name.split("_")[0].lower()
    if prefix in TARGET_MAP:
        cls = TARGET_MAP[prefix]
        final_counts[cls] += 1
        indexed_samples.append((wav, cls))

print(f"Total matched tracks: {len(indexed_samples)} / {len(all_wavs)}")
print("\nFinal 10-Class Distribution:")
for cls in sorted(set(TARGET_MAP.values())):
    print(f"  {cls:10s}: {final_counts[cls]} tracks")

Total matched tracks: 9758 / 9758

Final 10-Class Distribution:
  bass      : 1602 tracks
  drums     : 681 tracks
  flute     : 878 tracks
  guitar    : 798 tracks
  piano     : 1746 tracks
  sarod     : 554 tracks
  sitar     : 448 tracks
  tabla     : 644 tracks
  veena     : 636 tracks
  violin    : 1771 tracks


In [8]:
# Step 7: Check sample rate, duration, and channel format across classes
import soundfile as sf
import random

# Pick 1 sample file from each of the 10 classes
samples_by_class = {}
for wav_path, cls in indexed_samples:
    if cls not in samples_by_class:
        samples_by_class[cls] = wav_path
    if len(samples_by_class) == 10:
        break

print(f"{'Class':10s} | {'Sample Rate':11s} | {'Channels':8s} | {'Duration (s)':12s} | {'Subtype':10s}")
print("-" * 65)

for cls, wav_path in sorted(samples_by_class.items()):
    info = sf.info(wav_path)
    print(f"{cls:10s} | {info.samplerate:<11d} | {info.channels:<8d} | {info.duration:<12.2f} | {info.subtype:10s}")

Class      | Sample Rate | Channels | Duration (s) | Subtype   
-----------------------------------------------------------------
bass       | 44100       | 1        | 1.44         | PCM_24    
drums      | 44100       | 1        | 8.97         | PCM_16    
flute      | 44100       | 1        | 8.02         | PCM_16    
guitar     | 44100       | 1        | 1.84         | PCM_16    
piano      | 16000       | 1        | 4.00         | PCM_16    
sarod      | 48000       | 2        | 10.00        | PCM_16    
sitar      | 48000       | 2        | 10.00        | PCM_16    
tabla      | 44100       | 2        | 10.87        | PCM_16    
veena      | 48000       | 2        | 10.00        | PCM_16    
violin     | 44100       | 1        | 0.94         | PCM_16    


In [9]:
# Step 8: Define MFCC feature extractor and verify uniform tensor shapes
import librosa
import numpy as np
import soundfile as sf

TARGET_SR = 22050
DURATION_SEC = 2.0  # 2.0s window fits all instruments with minimal padding
TARGET_SAMPLES = int(TARGET_SR * DURATION_SEC)  # 44,100 samples
N_MFCC = 40
HOP_LENGTH = 512

def extract_features(audio_path, target_sr=TARGET_SR, target_samples=TARGET_SAMPLES, n_mfcc=N_MFCC, hop_length=HOP_LENGTH):
    # Load and resample to mono float32
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)

    # Pad if shorter than target_samples, slice if longer
    if len(y) < target_samples:
        pad_width = target_samples - len(y)
        y = np.pad(y, (0, pad_width), mode='constant')
    else:
        y = y[:target_samples]

    # Compute 40-band MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=target_sr, n_mfcc=n_mfcc, hop_length=hop_length)
    return mfcc.astype(np.float32)

print(f"{'Class':10s} | {'Raw Path (.wav)':25s} | {'Output MFCC Shape':20s} | {'Dtype':10s}")
print("-" * 72)

for cls, wav_path in sorted(samples_by_class.items()):
    mfcc_feat = extract_features(wav_path)
    print(f"{cls:10s} | {wav_path.name[:24]:25s} | {str(mfcc_feat.shape):20s} | {str(mfcc_feat.dtype):10s}")

Class      | Raw Path (.wav)           | Output MFCC Shape    | Dtype     
------------------------------------------------------------------------
bass       | Bass_56song.wav           | (40, 87)             | float32   
drums      | drum_110song.wav          | (40, 87)             | float32   
flute      | flute_222song.wav         | (40, 87)             | float32   
guitar     | guitar_417song.wav        | (40, 87)             | float32   
piano      | piano_469song.wav         | (40, 87)             | float32   
sarod      | sarod_389song.wav         | (40, 87)             | float32   
sitar      | sitar_256song.wav         | (40, 87)             | float32   
tabla      | tabla_19song.wav          | (40, 87)             | float32   
veena      | veena_106song.wav         | (40, 87)             | float32   
violin     | violin_1109song.wav       | (40, 87)             | float32   


In [10]:
# Step 9: Extract balanced MFCC dataset and cache to disk
import numpy as np
import random
from pathlib import Path

# Fix seed for deterministic dataset splits
random.seed(42)
np.random.seed(42)

SAMPLES_PER_CLASS = 100  # 100 samples * 10 classes = 1,000 samples (fast, balanced)
CLASSES = sorted(list(set(TARGET_MAP.values())))
class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES)}

# Group audio paths by target class
by_class = {cls: [] for cls in CLASSES}
for wav_path, cls in indexed_samples:
    by_class[cls].append(wav_path)

# Sample balanced tracks
selected_samples = []
for cls in CLASSES:
    available = by_class[cls]
    chosen = random.sample(available, min(SAMPLES_PER_CLASS, len(available)))
    for p in chosen:
        selected_samples.append((p, class_to_idx[cls]))

random.shuffle(selected_samples)

print(f"Extracting MFCC features for {len(selected_samples)} tracks across 10 classes...")
X_list = []
y_list = []

for i, (path, label) in enumerate(selected_samples):
    if (i + 1) % 200 == 0 or (i + 1) == len(selected_samples):
        print(f"  Processed {i + 1}/{len(selected_samples)} tracks...")
    feat = extract_features(path)
    X_list.append(feat)
    y_list.append(label)

X = np.stack(X_list, axis=0)  # Shape: (N, 40, 87)
y = np.array(y_list, dtype=np.int64)

# Train/Val Split (80% Train, 20% Val)
split_idx = int(0.8 * len(X))
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

# Save to scratch storage
np.save("/content/X_train.npy", X_train)
np.save("/content/y_train.npy", y_train)
np.save("/content/X_val.npy", X_val)
np.save("/content/y_val.npy", y_val)

print("\nFeature extraction complete!")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"y_train shape: {y_train.shape}, dtype: {y_train.dtype}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")

Extracting MFCC features for 1000 tracks across 10 classes...
  Processed 200/1000 tracks...
  Processed 400/1000 tracks...
  Processed 600/1000 tracks...
  Processed 800/1000 tracks...
  Processed 1000/1000 tracks...

Feature extraction complete!
X_train shape: (800, 40, 87), dtype: float32
y_train shape: (800,), dtype: int64
X_val shape:   (200, 40, 87)
y_val shape:   (200,)


In [11]:
# Step 10: PyTorch Dataset, DataLoader, and TimbreCNN Architecture
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 1. Dataset Wrapper
class IMIFeatureDataset(Dataset):
    def __init__(self, x_path, y_path):
        # Load cached numpy arrays: (N, 40, 87)
        self.X = torch.from_numpy(np.load(x_path)).float()
        self.y = torch.from_numpy(np.load(y_path)).long()
        # Add channel dimension: (N, 1, 40, 87)
        if self.X.ndim == 3:
            self.X = self.X.unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = IMIFeatureDataset("/content/X_train.npy", "/content/y_train.npy")
val_dataset = IMIFeatureDataset("/content/X_val.npy", "/content/y_val.npy")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 2. Lightweight 2D CNN Architecture
class TimbreCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 3. Dry-Run Verification on 1 Batch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TimbreCNN(num_classes=len(CLASSES)).to(device)

dummy_x, dummy_y = next(iter(train_loader))
dummy_x = dummy_x.to(device)
with torch.no_grad():
    logits = model(dummy_x)

print(f"Device: {device}")
print(f"Batch input shape:  {dummy_x.shape}")
print(f"Batch target shape: {dummy_y.shape}")
print(f"Logits shape:       {logits.shape} (Expected: [32, 10])")
print(f"Total parameters:   {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Device: cuda
Batch input shape:  torch.Size([32, 1, 40, 87])
Batch target shape: torch.Size([32])
Logits shape:       torch.Size([32, 10]) (Expected: [32, 10])
Total parameters:   94,410


In [12]:
# Step 11: Training Loop for 2D TimbreCNN (15 Epochs)
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

num_epochs = 15
best_val_acc = 0.0
save_path = Path("/content/timbre_cnn_best.pt")

print(f"{'Epoch':7s} | {'Train Loss':11s} | {'Train Acc':10s} | {'Val Loss':9s} | {'Val Acc':8s}")
print("-" * 55)

for epoch in range(1, num_epochs + 1):
    # Training Phase
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss = criterion(logits, y_b)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(y_b)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y_b).sum().item()
        train_total += len(y_b)

    train_loss /= train_total
    train_acc = 100.0 * train_correct / train_total

    # Validation Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            loss = criterion(logits, y_b)

            val_loss += loss.item() * len(y_b)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y_b).sum().item()
            val_total += len(y_b)

    val_loss /= val_total
    val_acc = 100.0 * val_correct / val_total
    scheduler.step(val_acc)

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        saved_mark = "  [*Best Saved]"
    else:
        saved_mark = ""

    print(f"{epoch:<7d} | {train_loss:<11.4f} | {train_acc:<9.1f}% | {val_loss:<9.4f} | {val_acc:<7.1f}%{saved_mark}")

print(f"\nTraining Complete! Best Validation Accuracy: {best_val_acc:.1f}%")
print(f"Model checkpoint saved to: {save_path}")

Epoch   | Train Loss  | Train Acc  | Val Loss  | Val Acc 
-------------------------------------------------------
1       | 1.9073      | 44.1     % | 1.6292    | 60.0   %  [*Best Saved]
2       | 1.3994      | 64.1     % | 1.1588    | 68.0   %  [*Best Saved]
3       | 1.0751      | 74.5     % | 0.9756    | 78.5   %  [*Best Saved]
4       | 0.9193      | 79.8     % | 0.8723    | 77.5   %
5       | 0.8053      | 80.6     % | 0.7529    | 82.0   %  [*Best Saved]
6       | 0.6841      | 84.2     % | 0.6726    | 80.5   %
7       | 0.6399      | 84.5     % | 0.6176    | 88.0   %  [*Best Saved]
8       | 0.5761      | 86.5     % | 0.5725    | 85.5   %
9       | 0.5288      | 86.9     % | 0.5576    | 85.5   %
10      | 0.4777      | 90.6     % | 0.4880    | 89.0   %  [*Best Saved]
11      | 0.4700      | 89.0     % | 0.4620    | 87.0   %
12      | 0.4374      | 90.6     % | 0.4420    | 88.0   %
13      | 0.3981      | 91.4     % | 0.4052    | 89.5   %  [*Best Saved]
14      | 0.3733      | 91.

In [13]:
# Step 12: Downstream Stem Purity & Bleed Audit
import torch
import torch.nn.functional as F
from pathlib import Path
import soundfile as sf
import numpy as np

# 1. Load best checkpoint into eval model
best_model = TimbreCNN(num_classes=len(CLASSES)).to(device)
best_model.load_state_dict(torch.load("/content/timbre_cnn_best.pt"))
best_model.eval()

# 2. Locate generated stems from Milestone 3
stem_dir = Path("/content/drive/MyDrive/sanidha_separated_samples")

# We evaluate: original mix, fine-tuned stems, baseline stems, and ground-truth stems
test_stems = [
    ("Original Polyphonic Mix", stem_dir / "01_original_mix.wav"),
    ("Ground-Truth Vocals",     stem_dir / "02_groundtruth_vocals.wav"),
    ("Ground-Truth Accomp",     stem_dir / "03_groundtruth_accompaniment.wav"),
    ("Baseline Vocals",         stem_dir / "04_baseline_vocals.wav"),
    ("Baseline Accomp",         stem_dir / "05_baseline_accompaniment.wav"),
    ("Fine-Tuned Vocals",       stem_dir / "06_finetuned_vocals.wav"),
    ("Fine-Tuned Accomp",       stem_dir / "07_finetuned_accompaniment.wav"),
]

def classify_wav(wav_path):
    if not wav_path.exists():
        return None
    # Extract features using our locked 22050 Hz / 40-band extractor
    feat = extract_features(str(wav_path))  # (40, 87)
    tensor_in = torch.from_numpy(feat).unsqueeze(0).unsqueeze(0).to(device) # (1, 1, 40, 87)
    with torch.no_grad():
        logits = best_model(tensor_in)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    return probs

print(f"{'Stem Name':30s} | {'Top Predicted Class':20s} | {'Top Prob':10s} | {'Percussion/Strings Bleed'}")
print("-" * 85)

for name, path in test_stems:
    probs = classify_wav(path)
    if probs is None:
        print(f"{name:30s} | [FILE NOT FOUND AT PATH]")
        continue
    top_idx = int(np.argmax(probs))
    top_class = CLASSES[top_idx]
    top_prob = probs[top_idx] * 100

    # Calculate energy attributed to classical Indian instruments
    tabla_prob = probs[class_to_idx["tabla"]] * 100
    veena_sitar = (probs[class_to_idx["sitar"]] + probs[class_to_idx["veena"]]) * 100
    bleed_summary = f"Tabla: {tabla_prob:.1f}%, Sitar/Veena: {veena_sitar:.1f}%"

    print(f"{name:30s} | {top_class:20s} | {top_prob:9.1f}% | {bleed_summary}")

Stem Name                      | Top Predicted Class  | Top Prob   | Percussion/Strings Bleed
-------------------------------------------------------------------------------------
Original Polyphonic Mix        | [FILE NOT FOUND AT PATH]
Ground-Truth Vocals            | [FILE NOT FOUND AT PATH]
Ground-Truth Accomp            | [FILE NOT FOUND AT PATH]
Baseline Vocals                | [FILE NOT FOUND AT PATH]
Baseline Accomp                | [FILE NOT FOUND AT PATH]
Fine-Tuned Vocals              | [FILE NOT FOUND AT PATH]
Fine-Tuned Accomp              | [FILE NOT FOUND AT PATH]


In [14]:
# Step 13: Mount Drive & Locate Separated .wav Files
from google.colab import drive
import os
from pathlib import Path

# 1. Ensure Google Drive is mounted
if not Path("/content/drive/MyDrive").exists():
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

# 2. Search for any generated stem files across Drive and local scratch
print("\nSearching for '06_finetuned_vocals.wav' or '01_original_mix.wav'...")

matches = list(Path("/content").glob("**/06_finetuned_vocals.wav")) + \
          list(Path("/content/drive/MyDrive").glob("**/06_finetuned_vocals.wav"))

if matches:
    stem_parent_dir = matches[0].parent
    print(f"\nSUCCESS: Found stems in folder: {stem_parent_dir}")
    print("Available files in this folder:")
    for f in sorted(stem_parent_dir.glob("*.wav")):
        print(f"  - {f.name}")
else:
    print("\nCould not find '06_finetuned_vocals.wav'. Searching for any recent .wav folders...")
    for p in Path("/content/drive/MyDrive").iterdir():
        if p.is_dir() and any(k in p.name.lower() for k in ["sanidha", "sample", "stem", "check"]):
            print(f"  Candidate folder: {p}")

Mounting Google Drive...
Mounted at /content/drive

Searching for '06_finetuned_vocals.wav' or '01_original_mix.wav'...

SUCCESS: Found stems in folder: /content/drive/MyDrive/sanidha_separated_samples
Available files in this folder:
  - 01_mix.wav
  - 02_groundtruth_vocals.wav
  - 03_groundtruth_accompaniment.wav
  - 04_baseline_vocals.wav
  - 05_baseline_accompaniment.wav
  - 06_finetuned_vocals.wav
  - 07_finetuned_accompaniment.wav


In [15]:
# Step 14: Downstream Stem Purity & Bleed Audit
import torch
import torch.nn.functional as F
from pathlib import Path
import numpy as np

# 1. Load best checkpoint into eval model
best_model = TimbreCNN(num_classes=len(CLASSES)).to(device)
best_model.load_state_dict(torch.load("/content/timbre_cnn_best.pt"))
best_model.eval()

# 2. Point to the verified folder on Google Drive
stem_dir = Path("/content/drive/MyDrive/sanidha_separated_samples")

test_stems = [
    ("Original Polyphonic Mix", stem_dir / "01_mix.wav"),
    ("Ground-Truth Vocals",     stem_dir / "02_groundtruth_vocals.wav"),
    ("Ground-Truth Accomp",     stem_dir / "03_groundtruth_accompaniment.wav"),
    ("Baseline Vocals",         stem_dir / "04_baseline_vocals.wav"),
    ("Baseline Accomp",         stem_dir / "05_baseline_accompaniment.wav"),
    ("Fine-Tuned Vocals",       stem_dir / "06_finetuned_vocals.wav"),
    ("Fine-Tuned Accomp",       stem_dir / "07_finetuned_accompaniment.wav"),
]

def classify_wav(wav_path):
    if not wav_path.exists():
        return None
    feat = extract_features(str(wav_path))  # (40, 87)
    tensor_in = torch.from_numpy(feat).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, 40, 87)
    with torch.no_grad():
        logits = best_model(tensor_in)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    return probs

print(f"{'Stem Name':28s} | {'Top Predicted Class':20s} | {'Top Prob':10s} | {'Percussion/Strings Bleed'}")
print("-" * 88)

for name, path in test_stems:
    probs = classify_wav(path)
    if probs is None:
        print(f"{name:28s} | [FILE NOT FOUND AT PATH]")
        continue
    top_idx = int(np.argmax(probs))
    top_class = CLASSES[top_idx]
    top_prob = probs[top_idx] * 100

    # Calculate energy attributed to classical Indian instruments
    tabla_prob = probs[class_to_idx["tabla"]] * 100
    veena_sitar = (probs[class_to_idx["sitar"]] + probs[class_to_idx["veena"]]) * 100
    bleed_summary = f"Tabla: {tabla_prob:.1f}%, Sitar/Veena: {veena_sitar:.1f}%"

    print(f"{name:28s} | {top_class:20s} | {top_prob:9.1f}% | {bleed_summary}")

Stem Name                    | Top Predicted Class  | Top Prob   | Percussion/Strings Bleed
----------------------------------------------------------------------------------------
Original Polyphonic Mix      | veena                |      56.6% | Tabla: 8.2%, Sitar/Veena: 80.3%
Ground-Truth Vocals          | violin               |      47.7% | Tabla: 0.0%, Sitar/Veena: 44.2%
Ground-Truth Accomp          | veena                |      37.7% | Tabla: 8.4%, Sitar/Veena: 52.0%
Baseline Vocals              | sitar                |      66.8% | Tabla: 0.0%, Sitar/Veena: 69.0%
Baseline Accomp              | guitar               |      38.8% | Tabla: 35.7%, Sitar/Veena: 15.4%
Fine-Tuned Vocals            | sitar                |      71.7% | Tabla: 0.0%, Sitar/Veena: 72.6%
Fine-Tuned Accomp            | veena                |      58.9% | Tabla: 5.6%, Sitar/Veena: 76.1%


In [16]:
# Step 15: Serialize Milestone 4 Audit Results to JSON
import json

m4_results = {
    "milestone": "Milestone 4 - Downstream Timbre Bleed Audit",
    "classifier_val_accuracy": float(best_val_acc),
    "classes": CLASSES,
    "evaluations": []
}

for name, path in test_stems:
    probs = classify_wav(path)
    if probs is None:
        continue
    top_idx = int(np.argmax(probs))
    m4_results["evaluations"].append({
        "stem_name": name,
        "top_class": CLASSES[top_idx],
        "top_probability": float(probs[top_idx]),
        "tabla_bleed_pct": float(probs[class_to_idx["tabla"]] * 100),
        "string_bleed_pct": float((probs[class_to_idx["sitar"]] + probs[class_to_idx["veena"]]) * 100),
        "full_distribution": {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}
    })

report_path = "/content/drive/MyDrive/milestone4_timbre_audit.json"
with open(report_path, "w") as f:
    json.dump(m4_results, f, indent=2)

print(f"Milestone 4 audit successfully saved to: {report_path}")

Milestone 4 audit successfully saved to: /content/drive/MyDrive/milestone4_timbre_audit.json


In [18]:
from google.colab import files

# Download the Milestone 4 audit report
files.download("/content/drive/MyDrive/milestone4_timbre_audit.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>